# Parameters, State, and Memory

Almost everything we do to a model other than calling it operates on its
*state*: the optimizer updates it, a checkpoint serializes it, device
placement moves it, fine-tuning trains part of it, and the answer to "will
this model fit on my GPU?" is a few lines of arithmetic over it. So far that
state has been handled for us; the layers created the tensors and the
training loop updated them. This section opens the box: how to reach any tensor in a model,
which tensors are trained and which merely travel with the model, what they
all cost in bytes, and how to share or freeze them.

In [ ]:
import jax
from jax import numpy as jnp
from flax import nnx
import optax

## Accessing Parameters

Our specimen is the residual MLP of that section,
redefined here so this section stands on its own: an input layer, a stack of
residual blocks, and an output head.

In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(self, d, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(d, d, rngs=rngs), nnx.relu,
            nnx.Linear(d, d, rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

rngs = nnx.Rngs(42)
net = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                     ResidualBlock(64, rngs), ResidualBlock(64, rngs),
                     nnx.Linear(64, 10, rngs=rngs))
X = jax.random.normal(jax.random.key(0), (2, 20))
net(X).shape

A model built from NNX modules is an object graph, and its parameters are
`nnx.Param` variables owned by the layers. To reach one, follow ordinary
attributes and container indices. `net.layers[3]` is the output layer and
its `bias` variable is marked trainable by its type.

In [ ]:
bias = net.layers[3].bias
type(bias), bias.shape

The same path syntax reaches arbitrarily deep. The first linear layer inside
the first residual block sits three levels down:

In [ ]:
net.layers[1].body.layers[0].kernel.shape

Gradients are not stored on parameters. `nnx.grad` returns a second state
tree with the same parameter paths, one gradient leaf per parameter:

In [ ]:
params = nnx.state(net, nnx.Param)
grads = nnx.grad(lambda model: model(X).sum())(net)
jax.tree_util.tree_structure(grads) == jax.tree_util.tree_structure(params)

Reaching parameters one path at a time is right for debugging a single layer.
The optimizer, weight decay, and checkpointing instead need *every* leaf, and
`nnx.state(net, nnx.Param)` selects them. Its flat view yields each array
together with the object-graph path that reaches it.

In [ ]:
[(path, leaf.shape) for path, leaf in params.flat_state()]

Read one of the paths closely.
`('layers', 1, 'body', 'layers', 0, 'kernel')` means: child 1 of `net`, the
first residual block, its `body`, child 0 of that sequence, and finally its
kernel. Paths survive arbitrary nesting. Filters select the kinds of state a
consumer needs, while checkpoints can save the full state
(that section):

In [ ]:
list(params)

One tree, one naming scheme, and every consumer, whether optimizer,
checkpoint, or debugger, walks it. This model has only `nnx.Param` state.
That is not always so.

## Parameters and Buffers

Some tensors must persist inside a model and follow it from device to device,
yet should never receive a gradient. The canonical example is batch
normalization [@Ioffe.Szegedy.2015]: each layer maintains a running mean
and variance of its inputs, updated during the forward pass and used at
prediction time. Those statistics must be saved with the model and must move
to the GPU with it, but the optimizer has no business touching them. A
precomputed positional table may be treated the same way. Other non-trained
state has a different lifetime: a language model's key--value cache belongs to
one generation request, so it follows the computation device but should not be
stored in a model checkpoint. Causal masks are often recomputed or registered
as non-persistent state for the same reason.

NNX uses `Variable` subclasses to distinguish kinds of state. `nnx.Param`
marks trainable state, `nnx.BatchStat` marks running statistics, and a custom
subclass can mark another persistent buffer. Filters select which kinds an
optimizer or checkpoint sees, so an ephemeral cache can remain outside the
checkpoint filter or be passed explicitly. Here is a module that standardizes its inputs
with statistics computed once from a reference sample:

In [ ]:
class Buffer(nnx.Variable):
    pass

class Whitener(nnx.Module):
    def __init__(self, mean, std, rngs):
        self.mean = Buffer(mean)
        self.std = Buffer(std)
        self.out = nnx.Linear(4, 2, rngs=rngs)

    def __call__(self, X):
        return self.out((X - self.mean) / self.std)

sample = jax.random.normal(jax.random.key(1), (100, 4)) * jnp.arange(1., 5.)
whiten = Whitener(sample.mean(0), sample.std(0), nnx.Rngs(2))
[(path, value.shape) for path, value in nnx.state(whiten).flat_state()]

Both kinds belong to the object graph and can be checkpointed together. The
optimizer selects `nnx.Param`, so it never sees the statistics.

In [ ]:
{'params': list(nnx.state(whiten, nnx.Param)),
 'buffers': list(nnx.state(whiten, Buffer))}

Wrapping an array in a `Variable` makes it state. We can see the resulting
tree operation on the CPU by splitting the graph, casting every state leaf,
and merging a low-precision clone. The original model remains unchanged:

In [ ]:
graph, state = nnx.split(whiten)
low = nnx.merge(graph, jax.tree.map(lambda x: x.astype(jnp.float16), state))
low.mean.dtype, whiten.mean.dtype

Device placement follows the same rule: `jax.device_put` moves a whole
state tree, parameters and buffers alike. Updating the model with that tree
moves all registered state. The same lines work on any host; on a machine
with an accelerator, `jax.devices()[0]` is a GPU or TPU:

In [ ]:
moved = jax.device_put(nnx.state(whiten), jax.devices()[0])
nnx.update(whiten, moved)
whiten.mean.device

## Counting Parameters, Counting Bytes

Before any training job comes the question of whether the model fits in
memory, and the answer is arithmetic you can do on a napkin. Counting
parameters is one line. Counting *bytes* requires remembering everything that
training keeps per parameter: the weight itself, its gradient, and the
optimizer's state. Adam maintains two running moments per parameter, so in
fp32 the ledger reads:

| Training state       | Precision | Bytes per parameter |
|----------------------|-----------|---------------------|
| Weights              | fp32      | 4                   |
| Gradients            | fp32      | 4                   |
| Adam first moment    | fp32      | 4                   |
| Adam second moment   | fp32      | 4                   |
| Total                |           | 16                  |

In [ ]:
params = nnx.state(net, nnx.Param)
leaves = jax.tree_util.tree_leaves(params)
n = sum(x.size for x in leaves)
weights = sum(x.size * x.dtype.itemsize for x in leaves)   # fp32: 4 bytes
grads, adam_state = weights, 2 * weights
print(f'{n} parameters: {(weights + grads + adam_state) / 2**20:.2f} MiB '
      'for weights + gradients + Adam state')

The optimizer state is real memory, allocated tensor by tensor. After a
single training step, Adam's two moments together hold exactly two extra
copies of the model:

In [ ]:
optimizer = nnx.Optimizer(net, optax.adam(1e-3), wrt=nnx.Param)
grads = nnx.grad(lambda model: model(X).sum())(net)
optimizer.update(net, grads)
moments = sum(x.size for x in jax.tree.leaves(optimizer.opt_state)
              if hasattr(x, 'ndim') and x.ndim > 0)
moments == 2 * n

For our little network the total is a third of a megabyte, which is why none
of this mattered until now. Scale the same arithmetic to a 1-billion-parameter
model and the standard fp32 Adam ledger gives a batch-independent floor: 4 GB
for the weights alone and 16 GB for weights, gradients, and optimizer state,
before storing a single activation (that section). Activations depend on
batch size, sequence length, and architecture; they can exceed parameter state
and are treated in that section.

Large models often train in mixed precision
[@Micikevicius.Narang.Alben.ea.2018], but storage policies differ. One
implementation convention counts 18 bytes per parameter (fp32 master weights,
an fp16 working copy, fp32 gradients, and two fp32 moments); the ZeRO paper
counts 16 by keeping gradients in fp16
[@Rajbhandari.Rasley.Ruwase.ea.2020]. Other bf16 and AMP stacks keep no
separate working copy or choose another dtype for optimizer state. State the
dtypes before quoting a multiplier. In the two conventions above, the fp32
Adam moments alone cost 8 bytes per parameter.


![Bytes per parameter under fp32 Adam training (top) versus a mixed-precision, ZeRO-style convention (bottom), drawn to the same width per byte. The two ledgers disagree on weights and gradients but agree, byte for byte, on the Adam moments m and v.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-memory-ledger.svg)

the figure lays the two conventions side by side, byte for
byte: the moments (right half of each bar) line up exactly, and the only
disagreement is how the weights and gradients themselves are stored.

## Tied Parameters

One tensor can serve several roles in a model. The standard example is the
two ends of a language model. The input embedding is a $|V| \times d$ table
mapping each of $|V|$ tokens to a $d$-dimensional vector; the output
projection maps a $d$-dimensional hidden state to $|V|$ logits, and its weight
matrix has the same shape and an analogous meaning: one vector per token.
*Tying* the two, using a single tensor for both roles, saves $|V| \times d$
parameters. The cited studies also found lower language-model perplexity in
their experimental settings
[@Press.Wolf.2017; @Inan.Khosravi.Socher.2017]. The savings are large: in
GPT-2 [@Radford.Wu.Child.ea.2019] the shared $50257 \times 768$ matrix is
about 38.6 million of the model's 124 million parameters, roughly 31%.

![One weight matrix W, two call sites: the embedding looks up a row by token id, the output head multiplies by its transpose to produce logits. Tying means both point at the same tensor, so gradients from both uses sum into one place.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-weight-tying.svg)

the figure sketches the picture behind the two call sites.

A miniature version shows the mechanics. `nnx.Embed` provides `attend`, which multiplies a hidden state by the
transpose of the embedding table, exactly the output projection. Calling it
reuses the module's one kernel, so the tying is visible in the state itself:
no head matrix exists.

In [ ]:
class TinyLM(nnx.Module):
    def __init__(self, vocab_size, d, tied=True, rngs=None):
        rngs = nnx.Rngs(0) if rngs is None else rngs
        self.tied = tied
        self.emb = nnx.Embed(vocab_size, d, rngs=rngs)
        self.body = nnx.Linear(d, d, rngs=rngs)
        self.head = None if tied else nnx.Linear(
            d, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, tokens):
        h = nnx.relu(self.body(self.emb(tokens)))
        if self.tied:
            return self.emb.attend(h)   # the embedding table, used as the head
        return self.head(h)

lm = TinyLM(vocab_size=100, d=16)
tokens = jax.random.randint(jax.random.key(1), (2, 8), 0, 100)
[(path, value.shape)
 for path, value in nnx.state(lm, nnx.Param).flat_state()]

There is no aliasing bookkeeping to get right, because the state contains
the table once. The parameter count reflects the $|V| \times d$ saving, and
an optimizer updates the table once per step:

In [ ]:
untied = TinyLM(vocab_size=100, d=16, tied=False)
(sum(x.size for x in jax.tree.leaves(nnx.state(lm, nnx.Param))),
 sum(x.size for x in jax.tree.leaves(nnx.state(untied, nnx.Param))))

A checkpoint of the tied model stores the table once, because the checkpoint
selects the model state. The flip side is that tied and untied checkpoints differ in
structure: the tied one has no head kernel, so loading it into an untied
module is a structural mismatch to resolve explicitly, not a silent aliasing
decision.

What about gradients? During backpropagation each use of the tensor
contributes a gradient, and the contributions accumulate into the gradient of
the single shared tensor. We can verify this against the untied twin: load
the tied model's values into it, run the same backward pass through both, and
the tied gradient equals the *sum* of the untied model's two gradients.

In [ ]:
untied.emb.embedding[...] = lm.emb.embedding
untied.body.kernel[...] = lm.body.kernel
untied.body.bias[...] = lm.body.bias
untied.head.kernel[...] = lm.emb.embedding.T
g_tied = nnx.grad(lambda model: model(tokens).sum())(lm)
g_untied = nnx.grad(lambda model: model(tokens).sum())(untied)
jnp.allclose(g_tied['emb']['embedding'],
             g_untied['emb']['embedding'] + g_untied['head']['kernel'].T)

## Freezing Parameters

Every fine-tuning recipe (that section) rests on one primitive,
and in NNX it lives in the optimizer: a leaf is frozen exactly when no update reaches it.
`optax.multi_transform` routes each leaf to a transformation chosen by a
label, and `optax.set_to_zero()` is the transformation that freezes, turning
every gradient it handles into a zero update. The typical pattern freezes a
pretrained backbone and trains only a new head. On a fresh copy of our
residual MLP, labeling everything but the output layer `'freeze'` leaves 650
of 18,634 parameters trainable:

In [ ]:
rngs = nnx.Rngs(3)
finetune = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                          ResidualBlock(64, rngs),
                          ResidualBlock(64, rngs),
                          nnx.Linear(64, 10, rngs=rngs))
ft_params = nnx.state(finetune, nnx.Param)
def labels(params):
    return jax.tree_util.tree_map_with_path(
        lambda path, _: ('train' if "['layers'][3]" in
                         jax.tree_util.keystr(path) else 'freeze'), params)

pure_params = nnx.to_pure_dict(ft_params)
pure_labels = labels(pure_params)
n_train = sum(x.size for x, l in zip(jax.tree_util.tree_leaves(pure_params),
                                     jax.tree_util.tree_leaves(pure_labels))
              if l == 'train')
(n_train, sum(x.size for x in jax.tree_util.tree_leaves(pure_params)))

The optimizer receives the whole tree plus the labels, and the partition does
the rest: Adam applies to the leaves labeled `'train'`, `set_to_zero` to the
others. One gradient step then moves the head and nothing else:

In [ ]:
tx = optax.multi_transform(
    {'train': optax.adam(0.1), 'freeze': optax.set_to_zero()}, labels)
optimizer = nnx.Optimizer(finetune, tx, wrt=nnx.Param)
before = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
grads = nnx.grad(lambda model: model(X).sum())(finetune)
optimizer.update(finetune, grads)
stepped = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
jax.tree_util.tree_map(lambda a, b: bool((a != b).any()), before, stepped)

Only the two leaves under `layers[3]`, the head's kernel and bias, changed.
Freezing traditionally carries two silent pitfalls, one about optimizer
memory and one about batch normalization; with explicit state, both become
questions you can settle by inspection.

In [ ]:
moments = sum(x.size for x in jax.tree_util.tree_leaves(optimizer.opt_state)
              if hasattr(x, 'ndim') and x.ndim > 0)
moments == 2 * n_train

Second, freezing governs `nnx.Param` state only. Batch normalization keeps
its running statistics as `nnx.BatchStat` variables and updates them during a
training-mode forward pass. Freeze the layer's scale and bias and the
statistics still drift because they never pass through the optimizer:

In [ ]:
bn = nnx.BatchNorm(4, use_running_average=False, momentum=0.9,
                   rngs=nnx.Rngs(4))
before = bn.mean[...]
bn(jax.random.normal(jax.random.key(5), (8, 4)) + 3)
bool(jnp.allclose(bn.mean, before))

The optimizer labels and `use_running_average` are orthogonal switches. The
first stops parameter updates; the second stops the forward pass from
updating running statistics. To pin a BatchNorm layer during fine-tuning you
need both. `nnx.view(model, use_running_average=True)` changes the mode on a
view of the full model.

Freezing whole tensors is the bluntest form of partial training.
Parameter-efficient methods instead add small trainable low-rank corrections
next to frozen weights; the linear algebra behind them is developed in
that section. A related idea maintains derived,
non-trained state: an *exponential moving average* (EMA) of the weights kept
alongside the trained ones and used for evaluation. Whether it improves on the
final iterate depends on its decay and the training schedule; controlled
examples appear in that section. Like BatchNorm statistics,
the average is state updated outside backpropagation.

In optax the average is one more piece of explicit state. `optax.ema` is a
transformation like any other: feed it the weights after each step, in place
of gradients, and its state carries an accumulator and a step count. With the
default debiasing, `ema.update` returns the corrected average. Continuing the
fine-tuning loop above, the average trails the moving head:

In [ ]:
ema = optax.ema(decay=0.9)
ema_state = ema.init(stepped)
for _ in range(5):
    grads = nnx.grad(lambda model: model(X).sum())(finetune)
    optimizer.update(finetune, grads)
    current = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
    avg, ema_state = ema.update(current, ema_state)
float(jnp.abs(avg['layers'][3]['bias']
              - current['layers'][3]['bias']).max())

## Summary

A model owns a typed state tree. `nnx.Param` marks trainable leaves;
`nnx.BatchStat` and custom `Variable` subclasses hold persistent state that
receives no optimizer updates. Filters select each view, and checkpoints can
save the whole tree. Training with Adam in fp32 costs 16 bytes per parameter
(4 weights, 4 gradients, 8 optimizer state) before activations.
Mixed-precision totals depend on the dtypes and copies the implementation
keeps. `nnx.Embed.attend` ties the embedding and the output head
structurally: one leaf in the tree, gradients summed over its uses. Freezing
is an optimizer-side partition, `optax.multi_transform` with `set_to_zero`;
it allocates no state for frozen leaves, but it does not stop `nnx.BatchStat`
updates in a training step.

## Exercises

1. Write a helper that reports the byte cost of fp32 Adam training separately
   for each top-level block of `net`. Which block
   dominates, and would that still hold if the residual blocks were 10 times
   wider?
1. BatchNorm's running mean and variance are non-trained state. Suppose you
   made them trainable parameters so that the optimizer updates them. What
   goes wrong during training, and why does gradient descent on a running
   average not compute the same thing as the forward-pass update rule it
   replaces?

3. Round-trip the tied model's `nnx.state(lm)` through serialization
   (that section) and reload it. Where, if anywhere, is the
   tying recorded? Explain why tying in NNX is a property of the module code
   rather than of the checkpoint.
4. In the tied `TinyLM`, try to freeze the embedding while training the head
   using `optax.multi_transform` labels. What choices do you have, and what
   does this teach about the interaction between tying and freezing?